# Carvana: understand one retained inventory snapshot

**Question:** which vehicles did the selected searches observe, and is their
coverage complete? Read the plan, coverage table, one source page and identity
checks in that order. Run All reads saved results without collection or writes.

The example is the retained **September 8, 2026** run, beginning with 42 Tesla
Model 3 listings from model year 2024. Trace VIN **5YJ3E1ET5RF828714**, listing
**4710782**, and its **$44,990 asking price** in the source and normalized tables.

## Settings

`PLAN_PATH` selects the saved query definitions; `RUN_PATH` selects their retained
run report. Keep these as a matched pair. Changing them selects local evidence;
it does not launch collection or change configuration files. A missing run means
inventory is unknown, not zero. The optional section at the end explains how the
historical POST requests worked.

This is a **historical replay of the selected saved run**, not today's inventory.
It uses the whole run's retained observation interval; there is no rolling current
cutoff in this lesson. `coverage` prints actual observation starts/ends and the
selected report's end time below. `observations` is the admitted vehicle table;
partial queries can have rows while coverage remains incomplete. An empty
`observations` table means no admitted evidence, not zero available cars.

Change only the matched `PLAN_PATH` / `RUN_PATH` pair to review another retained
run. There are no test/export switches or live calls to enable here. Learn in
order **00 -> 10 -> 11 -> 20 -> 24 -> 30**.


In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if (ROOT / "vehicle/src").is_dir():
    ROOT = ROOT / "vehicle"
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src/vehicle_tracker").is_dir():
    raise FileNotFoundError("Open from researchOS, vehicle, or vehicle/notebooks.")
sys.path.insert(0, str(ROOT / "src"))

from vehicle_tracker.carvana import parse_capture
from vehicle_tracker.readiness import query_readiness

PLAN_PATH = ROOT / "config/carvana_search_queries.json"
RUN_PATH = ROOT / "data/experiments/20260908_mvp1000_verified/run_report.json"
plan = json.loads(PLAN_PATH.read_text(encoding="utf-8"))

## 1. Inspect the query before the results

`plan["queries"]` contains the chosen partitions. The table exposes their filters and ZIPs. The next dictionary shows the request the collector would send for the first page; **displaying it does not send it**.

ZIP is the shopper's delivery/search context, not the vehicle's physical location. This saved sample disables location filtering. Different ZIPs can return the same vehicles, so their counts must not be added. These queries define an operational sample, not a representative or national inventory population.

In [ ]:
display(pd.DataFrame(plan["queries"])[["query_id", "zip_code", "location_filter", "filters"]])
query = plan["queries"][0]
request_example = dict(filters=query["filters"], pagination=dict(page=1, pageSize=24),
                       sortBy="MostPopular", zip5=query["zip_code"])
if query.get("location_filter", False):
    request_example["requestedFeatures"] = ["LocationBasedPrefiltering"]
print(json.dumps(request_example, indent=2))

## 2. Verify retained evidence and expose coverage

The next cell checks every artifact hash recorded in the plan checkpoint before admitting reports. Missing local experiments produce an explicit message. Changed or missing checkpoint artifacts block execution.

`query_readiness` calls `read_query_evidence` to verify retained page hashes, request context, admitted row counts and claimed completeness. It returns two tables: `coverage` (one row per query attempt, including unattempted queries) and `observations` (one row per admitted vehicle observation per query). Invalid evidence is shown in coverage and contributes no rows. Partial queries can contribute valid observations but remain incomplete. Multiple attempts remain separate.

A complete query must reconcile unique identities to the stable source total across consecutive pages and valid capture times. This certifies only that bounded query during its observation interval.

In [ ]:
report_paths = []
checkpoint = None
if RUN_PATH.is_file():
    checkpoint = json.loads(RUN_PATH.read_text(encoding="utf-8"))
    for outcome in checkpoint["outcomes"]:
        for filename, expected_hash in outcome.get("artifact_hashes", {}).items():
            artifact = Path(filename)
            if not artifact.is_file() or hashlib.sha256(artifact.read_bytes()).hexdigest() != expected_hash:
                raise ValueError(f"Retained artifact missing or changed: {artifact}")
        if outcome.get("report"):
            report_paths.append(Path(outcome["report"]))
else:
    print("No retained local run. Inventory is unknown, not zero.")

manifest = dict(queries=plan["queries"], population=plan["description"])
coverage, observations = query_readiness(manifest, report_paths)
display(coverage[["query_id", "status", "query_complete", "attempt_versions", "reported_total",
                  "admitted_rows", "reconciliation_difference", "observation_start", "observation_end", "reason"]])
print("All planned queries uniquely complete:", bool(
    coverage.query_complete.all() and coverage.attempt_versions.eq(1).all()))
print('Selected query plan:', PLAN_PATH)
print('Selected saved run:', RUN_PATH)
print('Historical run ended UTC:', checkpoint.get('ended_utc') if checkpoint else 'unavailable')


## 3. Trace one saved page into normalized columns

The first admitted observation identifies its report and retained page. `json_normalize` exposes source fields, while the pure `parse_capture` function produces the standard columns without network access or writes.

| Source field | Analysis column | Meaning / unit |
| --- | --- | --- |
| `vehicleId` | `listing_id` | Retailer listing identifier; key with `retailer` |
| `vin` | `vin` | Vehicle identity; keep listing aliases visible |
| `year`, `make`, `model` | Same names | Published model year and labels |
| `mileage` | `mileage_miles` | Published odometer, miles |
| `price.total` | `asking_price_usd` | Asking price in USD, not transaction price |
| `transportCost` | `transport_cost_usd` | Separate native transport charge in USD |
| `isPurchasePending` | `purchase_pending` | Native pending flag, not a completed sale |
| `vehicleLockType` | `vehicle_lock_type` | Native numeric code; no invented sale mapping |
| Capture timestamp | `observed_at_utc` | Observation time in UTC, not listing creation time |

Transport cost, pending and lock fields are extracted into explicit columns by `read_query_evidence`; the base parser also preserves native status in `card_text`. `availability_native` is missing for this source because it does not supply the older schema availability label. Zeros, negatives and missing values remain distinct.

In [ ]:
if not observations.empty:
    first = observations.iloc[0]
    report = json.loads(Path(first.report_path).read_text(encoding="utf-8"))
    page = next(p for p in report["pages"] if p["source_sha256"] == first.capture_id)
    source_path = Path(page["retained_source"])
    source = json.loads(source_path.read_text(encoding="utf-8"))
    print("Retained source:", source_path)
    print("Observed UTC:", source["captured_at_utc"])
    display(pd.DataFrame([source["pagination"]]))
    display(pd.json_normalize(source["vehicles"]).head(5))
    parsed = parse_capture(source)
    display(parsed[["listing_id", "vin", "year", "make", "model", "mileage_miles",
                    "asking_price_usd", "availability_native", "card_text"]].head(5))
else:
    print("No admitted observations to trace. Inspect coverage above.")

## 4. Check identities and missing values before using counts

`isna().sum()` reports missing fields without filling them. `groupby` shows row counts and unique identities per query. `duplicated(..., keep=False)` displays every overlapping listing membership instead of silently dropping one. The two identity checks expose listing IDs tied to multiple VINs and VINs tied to multiple listing IDs.

The final union counts distinct `(retailer, listing_id)` pairs for this sample only. It does not select a preferred price from duplicate observations, resolve VIN aliases, or certify inventory coverage. All source observations remain in `observations`.

In [ ]:
if not observations.empty:
    display(observations[["vin", "year", "mileage_miles", "asking_price_usd",
                          "purchase_pending", "vehicle_lock_type", "transport_cost_usd"]]
            .isna().sum().rename("missing_values").to_frame())
    display(observations.groupby("query_id").agg(
        observations=("listing_id", "size"), unique_listings=("listing_id", "nunique"),
        unique_vins=("vin", "nunique")))
    duplicates = observations[observations.duplicated(["retailer", "listing_id"], keep=False)]
    display(duplicates[["query_id", "retailer", "listing_id", "vin", "observed_at_utc", "report_path"]])
    listing_vins = observations.groupby(["retailer", "listing_id"]).vin.nunique()
    vin_listings = observations.groupby(["retailer", "vin"]).listing_id.nunique()
    display(listing_vins[listing_vins.gt(1)].rename("conflicting_vins").to_frame())
    display(vin_listings[vin_listings.gt(1)].rename("listing_ids_for_vin").to_frame())
    print("Observed sample union:", len(observations[["retailer", "listing_id"]].drop_duplicates()))
    print("Capture interval UTC:", observations.observed_at_utc.min(), "to", observations.observed_at_utc.max())
    print("National inventory coverage: UNVERIFIED. No sales count is produced.")

### Try it: a row is not the whole query

Find VIN `5YJ3E1ET5RF828714` in the displayed source and normalized rows. Verify
its listing ID **4710782** and asking price **44990**. Then inspect the first
query's `reported_total` and `admitted_rows`: the retained example has **42**
matching listings across two pages, even though the first page has **24** rows.
Use `observations.loc[observations.vin.eq('5YJ3E1ET5RF828714')]` to inspect the row.
An empty match means this VIN is not in the selected evidence; it does not mean a sale.

In Notebook 11, changing make/model/year/ZIP changes a request preview. Here,
changing the file paths changes the evidence you read. Neither action alone collects data.


## What to conclude from this snapshot

`coverage` establishes which declared queries completed and when. `observations`
contains their admitted rows; the identity and missing-field tables explain what
can be counted. A successful query covers that search during its capture window,
not national inventory. Asking prices and native pending flags remain observations.

Continue to [Notebook 11](11_carvana_live_collection_lab.ipynb) for the explicit one-page learning lab,
then [Notebook 20](20_carvana_history_analysis.ipynb) for daily collection
health, comparable VIN changes and matched prices. The [daily-cycle guide](../docs/daily_cycles.md)
explains collection windows and recovery; [the historical collection evaluation](../docs/collection_evaluation_20260908.md)
retains earlier method comparisons.

### Optional implementation references

[history.py](../src/vehicle_tracker/history.py) and [readiness.py](../src/vehicle_tracker/readiness.py)
read and validate the evidence used above. [carvana.py](../src/vehicle_tracker/carvana.py)
parses fields. The separate collection path is [collect_carvana_search.py](../scripts/collect_carvana_search.py),
[search_plan.py](../src/vehicle_tracker/search_plan.py), [search.py](../src/vehicle_tracker/search.py)
and [storage.py](../src/vehicle_tracker/storage.py). The historical walkthrough below
explains that path; its snippets are documentation, not executable notebook cells.


## Optional historical walkthrough: how the 42 Tesla listings were collected

Think of selecting **Tesla -> Model 3 -> 2024** on a car-search website and clicking through its results. Our Python program sends the search instructions directly to the server that supplies those results. It does not open 42 separate vehicle webpages.

### 1. Python asks for the first page

This is the address receiving every search request:

```text
https://apik.carvana.io/merch/search/api/v2/search
```

The URL stays the same. The `request` dictionary tells it which vehicles and which page we want:

```python
request = {
    "filters": {
        "makes": [{"name": "Tesla", "parentModels": [{"name": "Model 3"}]}],
        "year": {"min": 2024, "max": 2024}
    },
    "pagination": {"page": 1, "pageSize": 24},
    "sortBy": "MostPopular",
    "zip5": "08542"
}

response = requests.post(ENDPOINT, json=request, ...)
data = response.json()
```

Read the first line as **"send this search to Carvana."** Read the second as **"turn Carvana's answer into a Python dictionary I can work with."** The `...` abbreviates timeout and header settings. These snippets explain the running collector; they are Markdown, not executable notebook cells.

### 2. Carvana's answer already contains the vehicle data

In the retained September 8, 2026 example, page 1 reported **42 matching vehicles over two pages** and contained **24 vehicle records**. The shortened structure below reconstructs selected saved fields; it is not the complete original HTTP response:

```python
data = {
    "inventory": {
        "pagination": {
            "currentPage": 1,
            "pageSize": 24,
            "totalMatchedInventory": 42,
            "totalMatchedPages": 2
        },
        "vehicles": [
            {
                "vehicleId": 4710782,
                "vin": "5YJ3E1ET5RF828714",
                "year": 2024,
                "make": "Tesla",
                "model": "Model 3",
                "mileage": 17120,
                "price": {"total": 44990.0}
            }
            # Another 23 vehicle records were returned on this page.
        ]
    }
}
```

**There is no second scrape needed to obtain these prices or VINs: they are already inside the POST response.** For example:

```python
vehicles = data["inventory"]["vehicles"]  # the list of 24 vehicle records
first_vehicle = vehicles[0]              # one record from that list
first_vehicle["price"]["total"]          # 44990.0
```

### 3. Python turns those records into table rows

Each vehicle dictionary becomes one row. Our parser gives the source fields consistent column names:

| Source field in the answer | Column in our table | First saved vehicle |
| --- | --- | --- |
| `vehicleId` | `listing_id` | 4710782 |
| `vin` | `vin` | 5YJ3E1ET5RF828714 |
| `year` | `year` | 2024 |
| `mileage` | `mileage_miles` | 17120 |
| `price.total` | `asking_price_usd` | 44990.0 |

The actual collector performs this transformation with:

```python
capture = project_response(data, request, observed_at=capture_time)
frame = parse_search_capture(capture)
```

`project_response` keeps the selected source fields and attaches the request and capture time. `parse_search_capture` checks the fields and builds a pandas DataFrame: **24 rows for this page**. The price is an asking price in USD; it is not a transaction price.

### 4. Python saves the page so it survives after the script ends

The response and DataFrame are initially only in memory. The collector writes two useful forms to your computer:

```python
retained = retain_capture(capture, destination / "raw")
store_capture(
    destination / "vehicle.sqlite",
    run_id=run_id,
    page_number=1,
    raw_file=retained,
)
```

- The JSON file under `raw/` preserves selected source fields, the request and observation time. It lets us check where a value came from later. It is not the entire original HTTP body.
- `vehicle.sqlite` is a local database file containing the vehicle rows and a record of the page attempt. `store_capture` reads and parses the saved JSON before inserting its rows.

In the actual implementation, the JSON is retained before parsing/coverage checks so failed evidence can also be kept. Vehicle rows are admitted to the database only after the checks pass; failed attempts are recorded separately.

### 5. Python requests page 2 and adds its 18 vehicles

The answer told us there were two pages. The collector waits according to its request pacing and repeats the same process with:

```python
request["pagination"]["page"] = 2
```

The URL, Tesla filters and ZIP stay the same. Only the requested page number changes.

| Request | Vehicles returned and saved | Running count of distinct listings |
| --- | ---: | ---: |
| Page 1 | 24 | 24 |
| Page 2 | 18 | 42 |

Both counts above come from the saved trial. The collector checks for repeated identities, changed totals and invalid data before adding a page. At the end, **42 distinct saved listings matched the source's 42-result total**, so this particular query was marked complete.

`run_report.json` records the pages, counts, times, file hashes and whether the query completed. Getting HTTP 200 alone would not establish completeness.

### 6. The collector repeats this for the next search

The plan file is just a list of searches, for example 2024 Model 3, then 2023 Model 3, then 2022 Model 3. `collect_plan` goes through that list; `collect_search` handles the pages within each search.

```text
Chosen search plan
    |
    +-- 2024 Tesla Model 3
    |       POST page 1 -> check and save 24 rows
    |       POST page 2 -> check and save 18 rows -> query complete
    |
    +-- 2023 Tesla Model 3
    |       POST page 1 -> check and save rows
    |       POST page 2 -> ... until that search finishes or stops
    |
    +-- next chosen search ...
```

**That repeated request/check/save process is the collection.** A sample target or request/time limit can stop the process early. Earlier saved rows remain useful observations, but unfinished searches remain incomplete.

### 7. The notebook reads what the collector already saved

The collector performs the live requests and file/database writes. **This notebook is the explanation and inspection step afterward.** Earlier in this notebook, `query_readiness` reads the saved reports and JSON pages, checks their evidence, and rebuilds the `observations` DataFrame without another POST. That is why Run All can show vehicles while remaining offline.

For this example, the evidence is in [the saved 2024 Tesla query report](../data/experiments/20260908_mvp1000_verified/tesla_model3_2024/run_report.json), with its linked source files. These are historical observations, not today's inventory. A new day's collection must make fresh requests and use a new run destination.

A complete Tesla query is still only that selected search. The ZIP is shopper context, this request omits location filtering, and neither the 42-result reconciliation nor adding more searches proves complete national coverage. Missing listings are not confirmed sales.